# Temporal Transformer Networks for Longitudinal Alzheimer's Progression Prediction

**Janani Vaiyapuriappan**  
Department of Biomedical Engineering, Johns Hopkins University  
Course: Deep Learning for Medical Imaging (EN.580.745) · Supervisor: Prof. Ali Uneri

---

### Notebook Structure
1. Setup & Data Extraction
2. Dataset Exploration (EDA)
3. MRI Visualization
4. Data Pipeline (Preprocessing + DataLoader)
5. Model Architecture (SpatialEncoder + TemporalTransformer)
6. Task 1 — Subject-Level Conversion Prediction (Training + Evaluation)
7. Task 2 — Visit-Level CDR Worsening Prediction (Training + Evaluation)

## 1. Setup & Data Extraction

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tarfile

oasis_path = "/content/drive/MyDrive/DLMI ALZ Project/oasis-2"

# Only run this cell once — skip if already extracted
for part in ["OAS2_RAW_PART1.tar.gz", "OAS2_RAW_PART2.tar.gz"]:
    print(f"Extracting {part}...")
    with tarfile.open(f"{oasis_path}/{part}", "r:gz") as tar:
        tar.extractall(oasis_path)
    print(f"Completed extracting {part}")

print("All extracted!")

## 2. Dataset Exploration (EDA)

In [ ]:
import pandas as pd

oasis_path = "/content/drive/MyDrive/DLMI ALZ Project/oasis-2"

df = pd.read_excel(f"{oasis_path}/oasis_longitudinal_demographics.xlsx")
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print("\n--- CDR Distribution ---")
print(df['CDR'].value_counts())
print("\n--- Group Distribution ---")
print(df['Group'].value_counts())
print("\n--- Unique Subjects ---")
print(f"Unique subjects: {df['Subject ID'].nunique()}")
print("\n--- Sessions per Subject ---")
print(df.groupby('Subject ID')['Visit'].count().value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('OASIS-2 Dataset Overview', fontsize=14, fontweight='bold')

# Plot 1: CDR Distribution
cdr_counts = df['CDR'].value_counts().sort_index()
axes[0].bar(cdr_counts.index.astype(str), cdr_counts.values,
            color=['#2196F3', '#FF9800', '#F44336', '#9C27B0'],
            edgecolor='white', linewidth=0.5)
axes[0].set_title('CDR Score Distribution\n(all visits)', fontweight='bold')
axes[0].set_xlabel('CDR Score')
axes[0].set_ylabel('Number of Sessions')
for i, v in enumerate(cdr_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10, fontweight='bold')

# Plot 2: Group Distribution
group_counts = df['Group'].value_counts()
colors_group = ['#4CAF50', '#F44336', '#FF9800']
axes[1].bar(group_counts.index, group_counts.values,
            color=colors_group[:len(group_counts)],
            edgecolor='white', linewidth=0.5)
axes[1].set_title('Group Distribution\n(all sessions)', fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Number of Sessions')
for i, v in enumerate(group_counts.values):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=10, fontweight='bold')

# Plot 3: Sessions per subject
sessions_per_subj = df.groupby('Subject ID')['Visit'].count()
visit_dist = sessions_per_subj.value_counts().sort_index()
axes[2].bar(visit_dist.index.astype(str), visit_dist.values,
            color='#607D8B', edgecolor='white', linewidth=0.5)
axes[2].set_title('Sessions per Subject\n(longitudinal depth)', fontweight='bold')
axes[2].set_xlabel('Number of Sessions')
axes[2].set_ylabel('Number of Subjects')
for i, v in enumerate(visit_dist.values):
    axes[2].text(i, v + 0.3, str(v), ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{oasis_path}/dataset_overview.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: dataset_overview.png")

In [ ]:
# CDR longitudinal trajectories: converters vs stable

converter_ids = df[df['Group'] == 'Converted']['Subject ID'].unique()
demented_ids  = df[df['Group'] == 'Demented']['Subject ID'].unique()
stable_ids = [
    sid for sid in df[df['Group'] == 'Nondemented']['Subject ID'].unique()
    if sid not in converter_ids and sid not in demented_ids
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CDR Trajectories Over Time — OASIS-2', fontsize=13, fontweight='bold')

# Converters
ax = axes[0]
for sid in converter_ids[:15]:
    subj_data = df[df['Subject ID'] == sid].sort_values('Visit')
    if len(subj_data) >= 2:
        ax.plot(subj_data['Visit'], subj_data['CDR'],
                marker='o', linewidth=1.5, alpha=0.7, color='#E53935', markersize=4)
ax.set_title(f'Converters (n={len(converter_ids)})\nCDR progression over visits', fontweight='bold')
ax.set_xlabel('Visit Number')
ax.set_ylabel('CDR Score')
ax.set_yticks([0, 0.5, 1, 2])
ax.set_yticklabels(['0\n(Normal)', '0.5\n(Very Mild)', '1\n(Mild)', '2\n(Moderate)'])
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.set_facecolor('#fff5f5')
ax.grid(True, alpha=0.3)

# Stable
ax = axes[1]
for sid in stable_ids[:15]:
    subj_data = df[df['Subject ID'] == sid].sort_values('Visit')
    if len(subj_data) >= 2:
        ax.plot(subj_data['Visit'], subj_data['CDR'],
                marker='o', linewidth=1.5, alpha=0.7, color='#1E88E5', markersize=4)
ax.set_title(f'Stable Nondemented (sample of {min(15, len(stable_ids))})\nCDR stable over visits',
             fontweight='bold')
ax.set_xlabel('Visit Number')
ax.set_ylabel('CDR Score')
ax.set_yticks([0, 0.5, 1, 2])
ax.set_yticklabels(['0\n(Normal)', '0.5\n(Very Mild)', '1\n(Mild)', '2\n(Moderate)'])
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.set_facecolor('#f5f8ff')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{oasis_path}/cdr_trajectories.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cdr_trajectories.png")

## 3. MRI Visualization

In [ ]:
!pip install nibabel -q

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# Single subject — 3-plane view
sample_scan = f"{oasis_path}/OAS2_RAW_PART1/OAS2_0001_MR1/RAW/mpr-1.nifti.hdr"
img = nib.load(sample_scan)
data = img.get_fdata()

print("Shape:", data.shape)
print("Min/Max:", data.min(), data.max())

plt.figure(figsize=(12, 4))
for i, axis in enumerate([0, 1, 2]):
    plt.subplot(1, 3, i+1)
    slc = [slice(None)] * 3
    slc[axis] = data.shape[axis] // 2
    plt.imshow(data[tuple(slc)], cmap='gray')
    plt.title(f"{'Sagittal' if axis==0 else 'Coronal' if axis==1 else 'Axial'} (mid-slice)")
    plt.axis('off')
plt.suptitle("OAS2_0001_MR1 — T1w MRI")
plt.tight_layout()
plt.show()

In [ ]:
import os

def get_first_scan(subject_id, part):
    """Return path to first MRI scan for a subject, or None."""
    path = f"{oasis_path}/OAS2_RAW_{part}/{subject_id}_MR1/RAW/mpr-1.nifti.hdr"
    return path if os.path.exists(path) else None

def load_mid_slice(path):
    """Load NIfTI and return normalized axial mid-slice."""
    img = nib.load(path).get_fdata().squeeze()
    mid = img.shape[2] // 2
    slc = img[:, :, mid]
    return (slc - slc.min()) / (slc.max() - slc.min() + 1e-8)

# 3 converters vs 3 stable subjects — baseline scan comparison
conv_subjects   = df[df['Group'] == 'Converted']['Subject ID'].unique()[:3]
stable_subjects = [sid for sid in df[df['Group'] == 'Nondemented']['Subject ID'].unique()
                   if sid not in converter_ids][:3]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Baseline MRI Axial Slices — OASIS-2\n'
             'Top: MCI Converters (red) | Bottom: Stable Nondemented (blue)',
             fontsize=13, fontweight='bold')

for col, subj_id in enumerate(conv_subjects):
    path = next((get_first_scan(subj_id, p) for p in ['PART1', 'PART2']
                 if get_first_scan(subj_id, p)), None)
    if path:
        cdr_val = df[df['Subject ID'] == subj_id].sort_values('Visit').iloc[0]['CDR']
        axes[0, col].imshow(np.rot90(load_mid_slice(path)), cmap='gray')
        axes[0, col].set_title(f'{subj_id}\nBaseline CDR: {cdr_val}', fontsize=9)
        for spine in axes[0, col].spines.values():
            spine.set_edgecolor('#E53935'); spine.set_linewidth(3)
    else:
        axes[0, col].text(0.5, 0.5, 'Not found', ha='center', va='center')
    axes[0, col].axis('off')

for col, subj_id in enumerate(stable_subjects):
    path = next((get_first_scan(subj_id, p) for p in ['PART1', 'PART2']
                 if get_first_scan(subj_id, p)), None)
    if path:
        cdr_val = df[df['Subject ID'] == subj_id].sort_values('Visit').iloc[0]['CDR']
        axes[1, col].imshow(np.rot90(load_mid_slice(path)), cmap='gray')
        axes[1, col].set_title(f'{subj_id}\nBaseline CDR: {cdr_val}', fontsize=9)
        for spine in axes[1, col].spines.values():
            spine.set_edgecolor('#1E88E5'); spine.set_linewidth(3)
    else:
        axes[1, col].text(0.5, 0.5, 'Not found', ha='center', va='center')
    axes[1, col].axis('off')

fig.text(0.01, 0.75, 'Converters', va='center', rotation='vertical',
         fontsize=11, fontweight='bold', color='#E53935')
fig.text(0.01, 0.27, 'Stable', va='center', rotation='vertical',
         fontsize=11, fontweight='bold', color='#1E88E5')

plt.tight_layout()
plt.savefig(f"{oasis_path}/mri_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: mri_comparison.png")

## 4. Data Pipeline

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split

def get_mri_path(mri_id):
    for part in ["OAS2_RAW_PART1", "OAS2_RAW_PART2"]:
        path = f"{oasis_path}/{part}/{mri_id}/RAW/mpr-1.nifti.hdr"
        if os.path.exists(path):
            return path
    return None

def load_and_preprocess(path, target_shape=(64, 64, 64)):
    img = nib.load(path).get_fdata().squeeze()          # (256, 256, 128)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)  # normalize [0,1]
    factors = [t / s for t, s in zip(target_shape, img.shape)]
    img = zoom(img, factors, order=1)                   # downsample to 64^3
    return img.astype(np.float32)

# Build subject registry — label=1 if subject ever converts
subjects = []
for subj_id, group in df.groupby('Subject ID'):
    group = group.sort_values('Visit')
    label = 1 if (group['Group'] == 'Converted').any() else 0
    mri_paths = [get_mri_path(mri_id) for mri_id in group['MRI ID']]
    mri_paths = [p for p in mri_paths if p is not None]
    if len(mri_paths) >= 2:
        subjects.append({'subject_id': subj_id, 'paths': mri_paths,
                         'label': label, 'n_visits': len(mri_paths)})

print(f"Subjects with ≥2 sessions: {len(subjects)}")
print(f"Converters: {sum(s['label'] for s in subjects)}")
print(f"Non-converters: {sum(1 - s['label'] for s in subjects)}")

In [ ]:
from collections import Counter

conv_vc    = Counter(s['n_visits'] for s in subjects if s['label'] == 1)
nonconv_vc = Counter(s['n_visits'] for s in subjects if s['label'] == 0)
print("Converter visit distribution:",     dict(sorted(conv_vc.items())))
print("Non-converter visit distribution:", dict(sorted(nonconv_vc.items())))
print(f"Max visits: {max(s['n_visits'] for s in subjects)}")

In [ ]:
class OASISDataset(Dataset):
    def __init__(self, subjects, max_seq_len=5, target_shape=(64, 64, 64)):
        self.subjects = subjects
        self.max_seq_len = max_seq_len
        self.target_shape = target_shape

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subj = self.subjects[idx]
        scans = [load_and_preprocess(p, self.target_shape)
                 for p in subj['paths'][:self.max_seq_len]]
        # Pad to max_seq_len with zeros
        while len(scans) < self.max_seq_len:
            scans.append(np.zeros(self.target_shape, dtype=np.float32))
        scans = np.stack(scans, axis=0)               # (T, D, H, W)
        return (torch.tensor(scans).unsqueeze(1),     # (T, 1, 64, 64, 64)
                torch.tensor(subj['label'], dtype=torch.float32))

# Stratified 80/20 split
train_subj, val_subj = train_test_split(
    subjects, test_size=0.2, random_state=42,
    stratify=[s['label'] for s in subjects]
)
print(f"Train: {len(train_subj)} | Val: {len(val_subj)}")
print(f"Train converters: {sum(s['label'] for s in train_subj)}")
print(f"Val converters:   {sum(s['label'] for s in val_subj)}")

train_dataset = OASISDataset(train_subj)
val_dataset   = OASISDataset(val_subj)

# Weighted sampler for class imbalance
labels_t      = [s['label'] for s in train_subj]
class_counts  = [labels_t.count(0), labels_t.count(1)]
weights       = [1.0 / class_counts[l] for l in labels_t]
sampler       = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=4, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False)

print("Dataset ready!")
x, y = next(iter(train_loader))
print(f"Batch shape: {x.shape} | Labels: {y}")

## 5. Model Architecture

In [ ]:
import torch
import torch.nn as nn

class SpatialEncoder(nn.Module):
    """3D CNN — encodes a single MRI volume into a feature vector."""
    def __init__(self, out_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 32, 3, padding=1), nn.BatchNorm3d(32), nn.ReLU(),
            nn.MaxPool3d(2),                         # 64 → 32
            nn.Conv3d(32, 64, 3, padding=1), nn.BatchNorm3d(64), nn.ReLU(),
            nn.MaxPool3d(2),                         # 32 → 16
            nn.Conv3d(64, 128, 3, padding=1), nn.BatchNorm3d(128), nn.ReLU(),
            nn.MaxPool3d(2),                         # 16 → 8
            nn.Conv3d(128, 256, 3, padding=1), nn.BatchNorm3d(256), nn.ReLU(),
            nn.AdaptiveAvgPool3d(1),                 # → 256×1×1×1
        )
        self.fc = nn.Linear(256, out_dim)

    def forward(self, x):                            # x: (B, 1, 64, 64, 64)
        x = self.encoder(x).flatten(1)              # (B, 256)
        return self.fc(x)                            # (B, out_dim)


class TemporalTransformer(nn.Module):
    """Transformer over the sequence of spatially encoded MRI volumes."""
    def __init__(self, embed_dim=256, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.spatial_encoder = SpatialEncoder(out_dim=embed_dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, 5, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=nhead,
            dim_feedforward=512, dropout=dropout, batch_first=True
        )
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier   = nn.Sequential(
            nn.Linear(embed_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x):                            # x: (B, T, 1, 64, 64, 64)
        B, T, C, D, H, W = x.shape
        x = x.view(B * T, C, D, H, W)               # encode each timepoint
        x = self.spatial_encoder(x)                 # (B*T, embed_dim)
        x = x.view(B, T, -1)                        # (B, T, embed_dim)
        x = x + self.pos_embedding[:, :T, :]        # add positional encoding
        x = self.transformer(x)                     # (B, T, embed_dim)
        return self.classifier(x[:, 0, :]).squeeze(-1)  # classify from first token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = TemporalTransformer().to(device)

# Forward pass sanity check
x_test, y_test = next(iter(train_loader))
out = model(x_test.to(device))
print(f"Output shape: {out.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Task 1 — Subject-Level Conversion Prediction
**Label:** 1 if subject ever converts from MCI to AD, 0 otherwise.  
**Input:** Sequence of all available MRI scans per subject (padded to 5 timepoints).

In [ ]:
import time
from sklearn.metrics import f1_score, roc_auc_score

pos_weight = torch.tensor([136 / 14]).to(device)   # non-converters / converters
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds    = (torch.sigmoid(out) > 0.5).float()
        correct  += (preds == y).sum().item()
        total    += y.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y  = x.to(device), y.to(device)
            out   = model(x)
            loss  = criterion(out, y)
            total_loss += loss.item()
            preds = (torch.sigmoid(out) > 0.5).float()
            correct  += (preds == y).sum().item()
            total    += y.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return total_loss / len(loader), correct / total, all_preds, all_labels

num_epochs   = 30
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    start = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = val_epoch(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f"{oasis_path}/best_model.pth")

    print(f"Epoch {epoch+1:02d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | "
          f"Time: {time.time()-start:.1f}s")

print("Training complete! Best model saved.")

In [ ]:
# Task 1 — Training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Task 1: Subject-Level Conversion — Training Curves', fontweight='bold')

ax1.plot(history['train_loss'], label='Train Loss', color='#E53935')
ax1.plot(history['val_loss'],   label='Val Loss',   color='#1E88E5')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Acc', color='#E53935')
ax2.plot(history['val_acc'],   label='Val Acc',   color='#1E88E5')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{oasis_path}/training_curves.png", dpi=150)
plt.show()
print("Saved: training_curves.png")

In [ ]:
# Task 1 — Evaluation on validation set
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

model.load_state_dict(torch.load(f"{oasis_path}/best_model.pth"))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in val_loader:
        x, y  = x.to(device), y.to(device)
        out   = model(x)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).float()
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

print(classification_report(all_labels, all_preds,
                             target_names=['Non-converter', 'Converter']))
try:
    print(f"ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}")
except:
    print("ROC-AUC: cannot compute (only one class predicted)")

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-conv', 'Conv'],
            yticklabels=['Non-conv', 'Conv'])
plt.title('Task 1: Confusion Matrix — Subject-Level Conversion')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{oasis_path}/confusion_matrix.png", dpi=150)
plt.show()
print("Saved: confusion_matrix.png")

In [ ]:
# Visualize one training batch — what the model sees

x_batch, y_batch = next(iter(train_loader))
B, T = x_batch.shape[0], x_batch.shape[1]
n_show = min(4, B)

fig, axes = plt.subplots(n_show, T, figsize=(T * 2.5, n_show * 2.5))
fig.suptitle('Training Batch — What the Spatiotemporal Transformer Sees\n'
             'Rows = subjects | Columns = timepoints',
             fontsize=12, fontweight='bold')

for i in range(n_show):
    label_str   = 'Converter' if y_batch[i].item() == 1 else 'Stable'
    label_color = '#E53935'   if y_batch[i].item() == 1 else '#1E88E5'
    for t in range(T):
        ax  = axes[i, t] if n_show > 1 else axes[t]
        vol = x_batch[i, t, 0].numpy()
        if vol.max() == 0:   # padded timepoint
            ax.set_facecolor('#f0f0f0')
            ax.text(0.5, 0.5, 'padded', ha='center', va='center',
                    fontsize=8, color='gray', transform=ax.transAxes)
        else:
            ax.imshow(np.rot90(vol[:, :, vol.shape[2] // 2]), cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_title(f'Visit {t+1}', fontsize=9, fontweight='bold')
    axes[i, 0].set_ylabel(f'Subject {i+1}\n{label_str}',
                          fontsize=8, color=label_color, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{oasis_path}/training_batch_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: training_batch_visualization.png")

## 7. Task 2 — Visit-Level CDR Worsening Prediction
**Label:** 1 if CDR worsens at the next visit, 0 if stable.  
**Input:** All MRI scans up to and including visit *i*, predict CDR at visit *i+1*.  
**Split:** By subject ID to prevent data leakage across visits.

In [ ]:
# Analyze CDR transitions

transitions = []
for subj_id, group in df.groupby('Subject ID'):
    group = group.sort_values('Visit').reset_index(drop=True)
    for i in range(len(group) - 1):
        transitions.append({
            'subject_id': subj_id,
            'from_visit': group.loc[i,   'Visit'],
            'to_visit':   group.loc[i+1, 'Visit'],
            'curr_cdr':   group.loc[i,   'CDR'],
            'next_cdr':   group.loc[i+1, 'CDR'],
            'delta':      group.loc[i+1, 'CDR'] - group.loc[i, 'CDR']
        })

trans_df = pd.DataFrame(transitions)
print(f"Total transitions: {len(trans_df)}")
print("\nCDR transition counts:")
print(trans_df.groupby(['curr_cdr', 'next_cdr']).size().reset_index(name='count'))
print("\nDelta distribution:")
print(trans_df['delta'].value_counts().sort_index())

In [ ]:
# Build transition-level samples

transition_samples = []
for subj_id, group in df.groupby('Subject ID'):
    group = group.sort_values('Visit').reset_index(drop=True)
    for i in range(len(group) - 1):
        label = 1 if group.loc[i+1, 'CDR'] > group.loc[i, 'CDR'] else 0
        paths = [get_mri_path(mid) for mid in group.loc[:i, 'MRI ID']]
        paths = [p for p in paths if p is not None]
        if len(paths) >= 1:
            transition_samples.append({
                'subject_id': subj_id, 'paths': paths, 'label': label,
                'curr_cdr': group.loc[i, 'CDR'], 'next_cdr': group.loc[i+1, 'CDR'],
                'visit': group.loc[i, 'Visit']
            })

print(f"Total transition samples: {len(transition_samples)}")
print(f"Worsening (label=1): {sum(s['label'] for s in transition_samples)}")
print(f"Stable    (label=0): {sum(1 - s['label'] for s in transition_samples)}")

In [ ]:
# Subject-level split (prevents data leakage across visits)

unique_subjs_trans = list(set(s['subject_id'] for s in transition_samples))
subj_labels_trans  = {
    sid: max(s['label'] for s in transition_samples if s['subject_id'] == sid)
    for sid in unique_subjs_trans
}

train_subjs_trans, val_subjs_trans = train_test_split(
    unique_subjs_trans, test_size=0.2, random_state=42,
    stratify=[subj_labels_trans[s] for s in unique_subjs_trans]
)

train_trans = [s for s in transition_samples if s['subject_id'] in train_subjs_trans]
val_trans   = [s for s in transition_samples if s['subject_id'] in val_subjs_trans]

print(f"Train: {len(train_trans)} samples | Worsening: {sum(s['label'] for s in train_trans)}")
print(f"Val:   {len(val_trans)} samples   | Worsening: {sum(s['label'] for s in val_trans)}")

train_dataset_trans = OASISDataset(train_trans)
val_dataset_trans   = OASISDataset(val_trans)

labels_trans       = [s['label'] for s in train_trans]
class_counts_trans = [labels_trans.count(0), labels_trans.count(1)]
weights_trans      = [1.0 / class_counts_trans[l] for l in labels_trans]
sampler_trans      = WeightedRandomSampler(weights_trans, num_samples=len(weights_trans),
                                            replacement=True)

train_loader_trans = DataLoader(train_dataset_trans, batch_size=4, sampler=sampler_trans)
val_loader_trans   = DataLoader(val_dataset_trans,   batch_size=4, shuffle=False)

x, y = next(iter(train_loader_trans))
print(f"Batch shape: {x.shape} | Labels: {y}")

In [ ]:
# Task 2 — Training

model_trans      = TemporalTransformer().to(device)
pos_weight_trans = torch.tensor([192 / 31]).to(device)
criterion_trans  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_trans)
optimizer_trans  = torch.optim.Adam(model_trans.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_trans  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_trans, patience=5, factor=0.5)

num_epochs        = 30
best_val_loss_trans = float('inf')
history_trans = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    start = time.time()
    train_loss, train_acc = train_epoch(model_trans, train_loader_trans,
                                         optimizer_trans, criterion_trans, device)
    val_loss, val_acc, _, _ = val_epoch(model_trans, val_loader_trans, criterion_trans, device)
    scheduler_trans.step(val_loss)

    history_trans['train_loss'].append(train_loss)
    history_trans['val_loss'].append(val_loss)
    history_trans['train_acc'].append(train_acc)
    history_trans['val_acc'].append(val_acc)

    if val_loss < best_val_loss_trans:
        best_val_loss_trans = val_loss
        torch.save(model_trans.state_dict(), f"{oasis_path}/best_model_trans.pth")

    print(f"Epoch {epoch+1:02d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | "
          f"Time: {time.time()-start:.1f}s")

print("Done! Best model saved.")

In [ ]:
# Task 2 — Training curves

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Task 2: Visit-Level CDR Worsening — Training Curves', fontweight='bold')

ax1.plot(history_trans['train_loss'], label='Train Loss', color='#E53935')
ax1.plot(history_trans['val_loss'],   label='Val Loss',   color='#1E88E5')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_trans['train_acc'], label='Train Acc', color='#E53935')
ax2.plot(history_trans['val_acc'],   label='Val Acc',   color='#1E88E5')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{oasis_path}/training_curves_trans.png", dpi=150)
plt.show()
print("Saved: training_curves_trans.png")

In [ ]:
# Task 2 — Evaluation

model_trans.load_state_dict(torch.load(f"{oasis_path}/best_model_trans.pth"))
model_trans.eval()

all_preds_t, all_labels_t, all_probs_t = [], [], []
with torch.no_grad():
    for x, y in val_loader_trans:
        x, y  = x.to(device), y.to(device)
        out   = model_trans(x)
        probs = torch.sigmoid(out)
        preds = (probs > 0.5).float()
        all_probs_t.extend(probs.cpu().numpy())
        all_preds_t.extend(preds.cpu().numpy())
        all_labels_t.extend(y.cpu().numpy())

print(classification_report(all_labels_t, all_preds_t,
                             target_names=['Stable', 'Worsening']))
try:
    print(f"ROC-AUC: {roc_auc_score(all_labels_t, all_probs_t):.4f}")
except:
    print("ROC-AUC: cannot compute (only one class predicted)")

cm_t = confusion_matrix(all_labels_t, all_preds_t)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_t, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Stable', 'Worsening'],
            yticklabels=['Stable', 'Worsening'])
plt.title('Task 2: Confusion Matrix — CDR Worsening')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig(f"{oasis_path}/confusion_matrix_trans.png", dpi=150)
plt.show()
print("Saved: confusion_matrix_trans.png")